In [ ]:
# pip install arxiv
import importlib
import json

arxiv = importlib.import_module("arxiv")

client = arxiv.Client(page_size=200, delay_seconds=3, num_retries=5)
queries = [
    'abs:"retrieval augmented generation"',
    'abs:"dense retrieval"',
    'abs:"semantic search"',
    'abs:"large language model" AND abs:retrieval',
]

docs = {}
for q in queries:
    search = arxiv.Search(
        query=f'({q}) AND (cat:cs.CL OR cat:cs.IR OR cat:cs.LG)',
        max_results=3000,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    for r in client.results(search):
        docs[r.entry_id] = {
            "id": r.entry_id,
            "title": r.title,
            "abstract": r.summary.replace("\n", " "),
            "year": r.published.year,
        }

with open("corpus.jsonl", "w") as f:
    for d in docs.values():
        f.write(json.dumps(d) + "\n")
        
print(len(docs))

In [11]:
import pandas as pd

df = pd.read_json("corpus.jsonl", lines=True)
df["n_words"] = df["abstract"].str.split().str.len()

print(df.shape)
print(df["n_words"].describe())
print(df["year"].value_counts().sort_index())
print("empty abstracts:", (df["abstract"].str.strip() == "").sum())
print("duplicate titles:", df["title"].str.lower().duplicated().sum())

print(df.sample(3, random_state=0)[["title", "abstract"]].to_string())

(5648, 5)
count    5648.000000
mean      169.970786
std        38.292621
min        36.000000
25%       144.000000
50%       168.000000
75%       193.000000
max       313.000000
Name: n_words, dtype: float64
year
2009       1
2010       1
2011       1
2012       6
2013       5
2014       6
2015       1
2016       5
2017       7
2018      17
2019      11
2020      20
2021      73
2022     133
2023     386
2024    1323
2025    2089
2026    1563
Name: count, dtype: int64
empty abstracts: 0
duplicate titles: 2
                                                                                                                        title                                                                                                                                                                                                                                                                                                                                                                           

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_json("corpus.jsonl", lines=True)
df = df.drop_duplicates(subset="title", keep="first").reset_index(drop=True)
df["text"] = df["title"] + ". " + df["abstract"]   # title helps retrieval
df.to_json("corpus_clean.jsonl", orient="records", lines=True)

print(df.nsmallest(3, "n_words")[["title", "abstract"]] if "n_words" in df else
      df.assign(n=df.abstract.str.split().str.len()).nsmallest(3, "n")[["title", "abstract"]])

# Baseline config: decide now, then freeze it
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True,
                      ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(df["text"])
print(X.shape)

def search_tfidf(query, k=5):
    q = vec.transform([query])
    scores = cosine_similarity(q, X).ravel()
    top = scores.argsort()[::-1][:k]
    return [(df.title[i], round(float(scores[i]), 3)) for i in top]

for q in ["how do models look things up instead of memorizing facts",
          "reducing made-up answers from language models"]:
    print("\n", q)
    for t, s in search_tfidf(q):
        print(f"  {s}  {t}")

                                                  title  \
30    Towards Comprehensive Vietnamese Retrieval-Aug...   
15    Observations on Building RAG Systems for Techn...   
3127  Dense Retrieval for Low Resource Languages -- ...   

                                               abstract  
30    This paper presents our contributions towards ...  
15    Retrieval augmented generation (RAG) for techn...  
3127  This paper reports some difficulties and some ...  
(5646, 81521)

 how do models look things up instead of memorizing facts
  0.102  Generative AI for Low-Carbon Artificial Intelligence of Things with Large Language Models
  0.098  Generative Dense Retrieval: Memory Can Be a Burden
  0.094  Provable Benefits of In-Tool Learning for Large Language Models
  0.082  Generative Cross-Modal Retrieval: Memorizing Images in Multimodal Language Models for Retrieval and Beyond
  0.068  Semantic Advertising

 reducing made-up answers from language models
  0.105  FilterRAG: Zero-Shot In

In [13]:
import json

queries = {
    "q1": "how do models look things up instead of memorizing facts",
    "q2": "splitting long documents into pieces before indexing",
    "q3": "retrieval for languages with little training data",
    "q4": "making retrieval faster and cheaper at scale",
    "q5": "tell me about RAG",
    "q6": "what is RAG in machine learning",
    "q7": "where is RAG used nowadays",
    "q8": "how do you evaluate RAG systems",
    "q9": "efficiency and performance of these",
    "q10": "improving retrieval for long documents",
    "q11": "handling ambiguous queries",
    "q12": "handling multiple answers",
    "q13": "coding help using these",
    "q14": "how is RAG different from transformers",
    "q15": "is RAG used in ChatGPT",
    "q16": "current problems with RAG",
    "q18": "intro to RAG",
    "q19": "brief me on this topic in about 50 words",
    "q20": "what is the main idea behind RAG",
    "q21": "rag is used in LLMs",
    "q22": "rag used in small & simple chatbots",
    "q23": "rag used in large & complex chatbots",\
    "q24": "rag future",
    "q25": "rag current developments",
    "q26": "rag comparison with time",
    "q27": "rag comparison with other methods",
    "q28": "handling conflicting retrieved sources",
    "q29": "privacy in rag",
    
}
json.dump(queries, open("queries.json", "w"), indent=2)

In [14]:
import os
print(os.path.exists("queries.json"))   # should print True

True


In [15]:
import json, os
import numpy as np

LABELS = "labels.json"   # {qid: {doc_idx: 0/1/2}}
labels = json.load(open(LABELS)) if os.path.exists(LABELS) else {}

if os.path.exists("queries.json"):
    queries = json.load(open("queries.json"))
else:
    queries = {
        "q1": "how do models look things up instead of memorizing facts",
        "q2": "reducing made-up answers from language models",
    }
    print("queries.json not found; using built-in evaluation queries.")

def label_pool(system_fn, k=10):
    """system_fn(query, k) -> list of doc indices. Judge unseen docs only."""
    for qid, q in queries.items():
        seen = labels.setdefault(qid, {})
        for i in system_fn(q, k):
            if str(i) in seen:
                continue
            print(f"\nQ: {q}\n[{i}] {df.title[i]}\n{df.abstract[i][:500]}")
            seen[str(i)] = int(input("0=no 1=partial 2=yes > "))
            json.dump(labels, open(LABELS, "w"))

def mrr_ndcg(system_fn, k=10):
    rr, nd = [], []
    for qid, q in queries.items():
        rel = labels.get(qid, {})
        ranked = system_fn(q, k)
        gains = [rel.get(str(i), 0) for i in ranked]
        first = next((r for r, g in enumerate(gains, 1) if g >= 1), None)
        rr.append(1 / first if first else 0)
        dcg = sum((2**g - 1) / np.log2(r + 1) for r, g in enumerate(gains, 1))
        ideal = sorted(rel.values(), reverse=True)[:k]
        idcg = sum((2**g - 1) / np.log2(r + 1) for r, g in enumerate(ideal, 1))
        nd.append(dcg / idcg if idcg else 0)
    return np.mean(rr), np.mean(nd), rr, nd

# TF-IDF returning indices
def tfidf_idx(q, k=10):
    s = cosine_similarity(vec.transform([q]), X).ravel()
    return s.argsort()[::-1][:k].tolist()

In [16]:
print(len(queries))

28


In [17]:
label_pool(tfidf_idx, k=10)

## Phase 2 begins here...

In [ ]:
print(len(queries))                   
print(mrr_ndcg(tfidf_idx, k=10))

28
(np.float64(0.894047619047619), np.float64(0.7714095822017999), [0.5, 1.0, 0.5, 1.0, 0.2, 0.3333333333333333, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [np.float64(0.5912352048230277), np.float64(0.928816472271901), np.float64(0.7825837962406652), np.float64(0.8619774920955796), np.float64(0.39924298909821465), np.float64(0.2252946948005501), np.float64(0.6934697509937862), np.float64(0.9499924818408542), np.float64(0.8597378704108128), np.float64(0.919751958403785), np.float64(0.713775703442906), np.float64(0.7695525628707883), np.float64(0.6836414412537968), np.float64(0.6724305254594914), np.float64(0.8175295903539445), np.float64(0.8275922902348687), np.float64(0.6628651660287607), np.float64(0.6779449956136055), np.float64(0.4041189217546733), np.float64(0.9475751034706889), np.float64(0.9918271804495938), np.float64(0.7384350115782968), np.float64(0.9133131009302992), np.float64(0.8995201827278202), np.float6

In [ ]:
mrr, ndcg, rr_tfidf, nd_tfidf = mrr_ndcg(tfidf_idx)
print(mrr, ndcg)

# worst queries: where TF-IDF fails most
for qid, s in sorted(zip(queries, nd_tfidf), key=lambda x: x[1])[:5]:
    print(round(s, 3), queries[qid])

0.894047619047619 0.7714095822017999
0.225 what is RAG in machine learning
0.399 tell me about RAG
0.404 what is the main idea behind RAG
0.591 how do models look things up instead of memorizing facts
0.663 intro to RAG


In [ ]:
for qid, q in queries.items():
    pos = sum(g >= 1 for g in labels.get(qid, {}).values())
    if pos < 2:
        print(qid, pos, q)

#Outputs q17 has 0 documents labelled 1 or 2. So query 17 was dropped.

In [ ]:
drop = ["what is RAG in machine learning", "tell me about RAG",
        "what is the main idea behind RAG"]
# These queries perform poorly and have few positive labels, so we drop them from the queries-set.
queries = {qid: q for qid, q in queries.items() if q not in drop}
json.dump(queries, open("queries.json", "w"), indent=2)
print(len(queries))

25


In [ ]:
df = pd.read_json("corpus_clean.jsonl", lines=True)
print(df.shape, "text" in df.columns)   # expect (5646, ...) True

(5646, 5) True


In [ ]:
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True,
                      ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(df["text"])

In [ ]:
label_pool(tfidf_idx, k=10)
mrr, ndcg, rr_tfidf, nd_tfidf = mrr_ndcg(tfidf_idx)
print(len(queries), mrr, ndcg)
json.dump({"mrr": mrr, "ndcg": ndcg, "per_query": dict(zip(queries, nd_tfidf))},
          open("phase1_results.json", "w"))

25 0.94 0.8228324678398784


### Averaged GloVe Vectors

In [20]:
# pip install gensim
%pip install -q "gensim>=4.3.3"

import re
import gensim.downloader as api

kv = api.load("glove-wiki-gigaword-300")  # ~400 MB, one-time download

# IDF weights from a unigram vectorizer
uni = TfidfVectorizer(stop_words="english", min_df=2).fit(df["text"])
idf = dict(zip(uni.get_feature_names_out(), uni.idf_))
idf_default = max(idf.values())
stop_words = uni.get_stop_words() or set()

def embed(text):
    tokens = [
        token for token in re.findall(r"[a-z]+", text.lower())
        if token in kv and token not in stop_words
    ]

    if not tokens:
        return np.zeros(kv.vector_size, dtype=np.float32)

    weights = np.array(
        [idf.get(token, idf_default) for token in tokens],
        dtype=np.float32,
    )
    vectors = np.array([kv[token] for token in tokens])
    vector = (vectors * weights[:, None]).sum(axis=0) / weights.sum()

    norm = np.linalg.norm(vector)
    return vector / norm if norm else vector

D = np.vstack([embed(text) for text in df["text"]])

def glove_idx(q, k=10):
    return (D @ embed(q)).argsort()[::-1][:k].tolist()

for q in [
    "how do models look things up instead of memorizing facts",
    "reducing made-up answers from language models",
]:
    print("\n", q)
    for i in glove_idx(q, 5):
        print("  ", df.title.iloc[i])



Note: you may need to restart the kernel to use updated packages.

 how do models look things up instead of memorizing facts
   SuRe: Summarizing Retrievals using Answer Candidates for Open-domain QA of LLMs
   Good/Evil Reputation Judgment of Celebrities by LLMs via Retrieval Augmented Generation
   Insights into LLM Long-Context Failures: When Transformers Know but Don't Tell
   Large Language Models and Multimodal Retrieval for Visual Word Sense Disambiguation
   Thought-Retriever: Don't Just Retrieve Raw Data, Retrieve Thoughts for Memory-Augmented Agentic Systems

 reducing made-up answers from language models
   Augmenting LLM Reasoning with Dynamic Notes Writing for Complex QA
   Ask Good Questions for Large Language Models
   Retrieval Augmented Question Answering: When Should LLMs Admit Ignorance?
   Adapting Standard Retrieval Benchmarks to Evaluate Generated Answers
   Learning by Analogy: Enhancing Few-Shot Prompting for Math Word Problem Solving with Computational Graph-Ba

In [22]:
import os
print(os.path.exists("glove_doc_vectors.npy"))
np.save("glove_doc_vectors.npy", D)

True


In [28]:
label_pool(glove_idx, k=10)
print(mrr_ndcg(glove_idx)[:2])
print(mrr_ndcg(tfidf_idx)[:2])


Q: improving retrieval for long documents
[3990] Vietnamese Legal Information Retrieval in Question-Answering System
In the modern era of rapidly increasing data volumes, accurately retrieving and recommending relevant documents has become crucial in enhancing the reliability of Question Answering (QA) systems. Recently, Retrieval Augmented Generation (RAG) has gained significant recognition for enhancing the capabilities of large language models (LLMs) by mitigating hallucination issues in QA systems, which is particularly beneficial in the legal domain. Various methods, such as semantic search using dense ve

Q: improving retrieval for long documents
[180] Retrieval Augmented Generation Evaluation for Health Documents
Safe and trustworthy use of Large Language Models (LLM) in the processing of healthcare documents and scientific papers could substantially help clinicians, scientists and policymakers in overcoming information overload and focusing on the most relevant information at 

ValueError: invalid literal for int() with base 10: ''

In [ ]:
def add_label(qid, doc_idx, grade):
    labels.setdefault(qid, {})[str(doc_idx)] = grade
    json.dump(labels, open(LABELS, "w"))

add_label("q25", 180, 2)